# SAR feature data set only 

In [1]:
import os, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.neighbors import BallTree

# ---------------- 1. LOAD & CLEAN DATA ----------------
DATA_PATH = "data/corn_2016_2023_processed.parquet" 
df = pd.read_parquet(DATA_PATH)

# Ensure no NaNs in target
df = df[df["yield"].notna()].copy()
print(f"Loaded {len(df)} rows with {len(df.columns)} columns.")

# ---------------- 2. DYNAMIC FEATURE GROUPING ----------------
SAR_RAW = [c for c in df.columns if (c.startswith("VV_") or c.startswith("VH_")) and not c.endswith("_2")]
SAR_INDICES = [c for c in df.columns if c.startswith(("PR_", "RVI_"))]
WEATHER = [c for c in df.columns if any(k in c for k in ["prcp_", "srad_", "gdd_"]) and "_season" not in c]
TERRAIN = ["DEM", "slope_deg", "aspect_deg", "shape_index"]
SOIL    = [c for c in df.columns if c.startswith("soil_")]

SAR_ALL = SAR_RAW + SAR_INDICES

# ---------------- 3. SPATIAL FEATURE GENERATION ----------------
print("Calculating Field-Level Spatial Features (Means)...")
field_means = df.groupby(['farm_name', 'Year'])[SAR_ALL].transform('mean').astype(np.float32)
field_means.columns = [f"{c}_field_mean" for c in field_means.columns]
df = pd.concat([df, field_means], axis=1)

SAR_SPATIAL = SAR_ALL + list(field_means.columns)

def add_spatial_neighbor_features(df, sar_cols, n_neighbors=8):
    print(f"Generating BallTree neighbor features for {len(sar_cols)} features...")
    neighbor_data = np.zeros((len(df), len(sar_cols)), dtype=np.float32)
    
    for farm in df['farm_name'].unique():
        farm_mask = df['farm_name'] == farm
        farm_df = df[farm_mask]
        if len(farm_df) <= n_neighbors: continue
            
        tree = BallTree(farm_df[['latitude', 'longitude']].values)
        _, indices = tree.query(farm_df[['latitude', 'longitude']].values, k=n_neighbors+1)
        
        for i, col in enumerate(sar_cols):
            vals = farm_df[col].values
            neighbor_data[farm_mask, i] = vals[indices[:, 1:]].mean(axis=1)
            
    new_cols = [f"{c}_neighbor_8" for c in sar_cols]
    return pd.concat([df, pd.DataFrame(neighbor_data, columns=new_cols, index=df.index)], axis=1), new_cols

df, SAR_NEIGHBORS = add_spatial_neighbor_features(df, SAR_ALL)

# ---------------- 4. CONFIGURATION ----------------
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 50))
RUN_MODE = "BASE" 

FEATURE_MAP = {
    "BASE":         SAR_ALL + ["Year", "latitude"],
    "WEATHER":      SAR_ALL + WEATHER + ["Year", "latitude"],
    "TERRAIN":      SAR_ALL + WEATHER + TERRAIN + SOIL + ["Year", "latitude"],
    "FULL_SPATIAL": SAR_SPATIAL + SAR_NEIGHBORS + WEATHER + TERRAIN + SOIL + ["Year", "latitude", "longitude"]
}

SELECTED_FEATURES = FEATURE_MAP[RUN_MODE]
X = df[SELECTED_FEATURES]
y = df["yield"].to_numpy(dtype=np.float32)
groups = df["farm_name"].astype(str).to_numpy()
years = df["Year"]

# Stratification for balanced folds
y_binned = pd.qcut(y, q=5, labels=False)

print(f"🚀 Mode: {RUN_MODE} | Features: {len(SELECTED_FEATURES)} | Device: H200 GPU")

# ---------------- 5. CV LOOP ----------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
overall_start = time.time()

for k, (tr, te) in enumerate(sgkf.split(X, y_binned, groups=groups), 1):
    f_start = time.time()
    
    X_tr, X_te = X.iloc[tr], X.iloc[te]
    y_tr, y_te = y[tr], y[te]
    
    # Year-Mean Normalization
    yr_map = df.iloc[tr].groupby("Year")["yield"].mean().to_dict()
    global_mean = y_tr.mean()
    mu_tr = years.iloc[tr].map(yr_map).fillna(global_mean).values
    mu_te = years.iloc[te].map(yr_map).fillna(global_mean).values

    # Model Definition (with Early Stopping in constructor for XGB 2.0+)
    model = xgb.XGBRegressor(
        tree_method="hist",
        device="cuda",
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.7,
        colsample_bytree=0.6,
        n_jobs=CPUS,
        random_state=42,
        reg_lambda=10.0,
        reg_alpha=1.0,
        early_stopping_rounds=50
    )
    
    model.fit(X_tr, y_tr / mu_tr, eval_set=[(X_te, y_te / mu_te)], verbose=False)
    
    p_ratio = model.predict(X_te)
    p_raw = p_ratio * mu_te

    # --- METRIC CALCULATIONS ---
    r2 = r2_score(y_te, p_raw)
    mae = mean_absolute_error(y_te, p_raw)
    rmse = np.sqrt(mean_squared_error(y_te, p_raw))
    
    y_range = np.max(y_te) - np.min(y_te)
    nrmse = (rmse / y_range) * 100 if y_range != 0 else 0
    mape = mean_absolute_percentage_error(y_te, p_raw) * 100
    
    z_true = np.where(y_te/mu_te < 0.9, 0, np.where(y_te/mu_te <= 1.1, 1, 2))
    z_pred = np.where(p_ratio < 0.9, 0, np.where(p_ratio <= 1.1, 1, 2))
    acc = (z_true == z_pred).mean()

    results.append({
        'R2': r2, 'MAE': mae, 'RMSE': rmse, 
        'NRMSE': nrmse, 'MAPE': mape, 'Zone_Acc': acc
    })

    print(f"Fold {k} | R2: {r2:.3f} | RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | Time: {time.time()-f_start:.1f}s")

# ---------------- 6. DETAILED SUMMARY ----------------
res_df = pd.DataFrame(results)

print(f"\n📊 FINAL SUMMARY: {RUN_MODE}")
print(f"Total Time: {time.time()-overall_start:.2f}s")
print("-" * 35)
print(f"R²:    {res_df['R2'].mean():.4f} ± {res_df['R2'].std():.4f}")
print(f"MAE:   {res_df['MAE'].mean():.2f}")
print(f"RMSE:  {res_df['RMSE'].mean():.2f}")
print(f"NRMSE: {res_df['NRMSE'].mean():.2f}%")
print(f"MAPE:  {res_df['MAPE'].mean():.2f}%")
print(f"Acc:   {res_df['Zone_Acc'].mean():.4f}")
print("-" * 35)


Loaded 2108996 rows with 323 columns.
Calculating Field-Level Spatial Features (Means)...
Generating BallTree neighbor features for 264 features...
🚀 Mode: BASE | Features: 266 | Device: H200 GPU
Fold 1 | R2: 0.449 | RMSE: 35.66 | MAPE: 18.81% | Time: 14.3s
Fold 2 | R2: 0.420 | RMSE: 34.66 | MAPE: 17.13% | Time: 7.9s
Fold 3 | R2: 0.361 | RMSE: 38.22 | MAPE: 19.35% | Time: 10.2s
Fold 4 | R2: 0.315 | RMSE: 40.82 | MAPE: 21.20% | Time: 7.2s
Fold 5 | R2: 0.428 | RMSE: 36.54 | MAPE: 19.18% | Time: 11.2s

📊 FINAL SUMMARY: BASE
Total Time: 53.23s
-----------------------------------
R²:    0.3947 ± 0.0551
MAE:   28.19
RMSE:  37.18
NRMSE: 11.17%
MAPE:  19.13%
Acc:   0.5291
-----------------------------------


# SAR feature data set with weather 

In [2]:
import os, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.neighbors import BallTree

# ---------------- 1. LOAD & CLEAN DATA ----------------
DATA_PATH = "data/corn_2016_2023_processed.parquet" 
df = pd.read_parquet(DATA_PATH)

# Ensure no NaNs in target
df = df[df["yield"].notna()].copy()
print(f"Loaded {len(df)} rows with {len(df.columns)} columns.")

# ---------------- 2. DYNAMIC FEATURE GROUPING ----------------
SAR_RAW = [c for c in df.columns if (c.startswith("VV_") or c.startswith("VH_")) and not c.endswith("_2")]
SAR_INDICES = [c for c in df.columns if c.startswith(("PR_", "RVI_"))]
WEATHER = [c for c in df.columns if any(k in c for k in ["prcp_", "srad_", "gdd_"]) and "_season" not in c]
TERRAIN = ["DEM", "slope_deg", "aspect_deg", "shape_index"]
SOIL    = [c for c in df.columns if c.startswith("soil_")]

SAR_ALL = SAR_RAW + SAR_INDICES

# ---------------- 3. SPATIAL FEATURE GENERATION ----------------
print("Calculating Field-Level Spatial Features (Means)...")
field_means = df.groupby(['farm_name', 'Year'])[SAR_ALL].transform('mean').astype(np.float32)
field_means.columns = [f"{c}_field_mean" for c in field_means.columns]
df = pd.concat([df, field_means], axis=1)

SAR_SPATIAL = SAR_ALL + list(field_means.columns)

def add_spatial_neighbor_features(df, sar_cols, n_neighbors=8):
    print(f"Generating BallTree neighbor features for {len(sar_cols)} features...")
    neighbor_data = np.zeros((len(df), len(sar_cols)), dtype=np.float32)
    
    for farm in df['farm_name'].unique():
        farm_mask = df['farm_name'] == farm
        farm_df = df[farm_mask]
        if len(farm_df) <= n_neighbors: continue
            
        tree = BallTree(farm_df[['latitude', 'longitude']].values)
        _, indices = tree.query(farm_df[['latitude', 'longitude']].values, k=n_neighbors+1)
        
        for i, col in enumerate(sar_cols):
            vals = farm_df[col].values
            neighbor_data[farm_mask, i] = vals[indices[:, 1:]].mean(axis=1)
            
    new_cols = [f"{c}_neighbor_8" for c in sar_cols]
    return pd.concat([df, pd.DataFrame(neighbor_data, columns=new_cols, index=df.index)], axis=1), new_cols

df, SAR_NEIGHBORS = add_spatial_neighbor_features(df, SAR_ALL)

# ---------------- 4. CONFIGURATION ----------------
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 50))
RUN_MODE = "WEATHER" 

FEATURE_MAP = {
    "BASE":         SAR_ALL + ["Year", "latitude"],
    "WEATHER":      SAR_ALL + WEATHER + ["Year", "latitude"],
    "TERRAIN":      SAR_ALL + WEATHER + TERRAIN + SOIL + ["Year", "latitude"],
    "FULL_SPATIAL": SAR_SPATIAL + SAR_NEIGHBORS + WEATHER + TERRAIN + SOIL + ["Year", "latitude", "longitude"]
}

SELECTED_FEATURES = FEATURE_MAP[RUN_MODE]
X = df[SELECTED_FEATURES]
y = df["yield"].to_numpy(dtype=np.float32)
groups = df["farm_name"].astype(str).to_numpy()
years = df["Year"]

# Stratification for balanced folds
y_binned = pd.qcut(y, q=5, labels=False)

print(f"🚀 Mode: {RUN_MODE} | Features: {len(SELECTED_FEATURES)} | Device: H200 GPU")

# ---------------- 5. CV LOOP ----------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
overall_start = time.time()

for k, (tr, te) in enumerate(sgkf.split(X, y_binned, groups=groups), 1):
    f_start = time.time()
    
    X_tr, X_te = X.iloc[tr], X.iloc[te]
    y_tr, y_te = y[tr], y[te]
    
    # Year-Mean Normalization
    yr_map = df.iloc[tr].groupby("Year")["yield"].mean().to_dict()
    global_mean = y_tr.mean()
    mu_tr = years.iloc[tr].map(yr_map).fillna(global_mean).values
    mu_te = years.iloc[te].map(yr_map).fillna(global_mean).values

    # Model Definition (with Early Stopping in constructor for XGB 2.0+)
    model = xgb.XGBRegressor(
        tree_method="hist",
        device="cuda",
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.7,
        colsample_bytree=0.6,
        n_jobs=CPUS,
        random_state=42,
        reg_lambda=10.0,
        reg_alpha=1.0,
        early_stopping_rounds=50
    )
    
    model.fit(X_tr, y_tr / mu_tr, eval_set=[(X_te, y_te / mu_te)], verbose=False)
    
    p_ratio = model.predict(X_te)
    p_raw = p_ratio * mu_te

    # --- METRIC CALCULATIONS ---
    r2 = r2_score(y_te, p_raw)
    mae = mean_absolute_error(y_te, p_raw)
    rmse = np.sqrt(mean_squared_error(y_te, p_raw))
    
    y_range = np.max(y_te) - np.min(y_te)
    nrmse = (rmse / y_range) * 100 if y_range != 0 else 0
    mape = mean_absolute_percentage_error(y_te, p_raw) * 100
    
    z_true = np.where(y_te/mu_te < 0.9, 0, np.where(y_te/mu_te <= 1.1, 1, 2))
    z_pred = np.where(p_ratio < 0.9, 0, np.where(p_ratio <= 1.1, 1, 2))
    acc = (z_true == z_pred).mean()

    results.append({
        'R2': r2, 'MAE': mae, 'RMSE': rmse, 
        'NRMSE': nrmse, 'MAPE': mape, 'Zone_Acc': acc
    })

    print(f"Fold {k} | R2: {r2:.3f} | RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | Time: {time.time()-f_start:.1f}s")

# ---------------- 6. DETAILED SUMMARY ----------------
res_df = pd.DataFrame(results)

print(f"\n📊 FINAL SUMMARY: {RUN_MODE}")
print(f"Total Time: {time.time()-overall_start:.2f}s")
print("-" * 35)
print(f"R²:    {res_df['R2'].mean():.4f} ± {res_df['R2'].std():.4f}")
print(f"MAE:   {res_df['MAE'].mean():.2f}")
print(f"RMSE:  {res_df['RMSE'].mean():.2f}")
print(f"NRMSE: {res_df['NRMSE'].mean():.2f}%")
print(f"MAPE:  {res_df['MAPE'].mean():.2f}%")
print(f"Acc:   {res_df['Zone_Acc'].mean():.4f}")
print("-" * 35)


Loaded 2108996 rows with 323 columns.
Calculating Field-Level Spatial Features (Means)...
Generating BallTree neighbor features for 264 features...
🚀 Mode: WEATHER | Features: 287 | Device: H200 GPU
Fold 1 | R2: 0.380 | RMSE: 37.85 | MAPE: 20.01% | Time: 18.8s
Fold 2 | R2: 0.374 | RMSE: 36.02 | MAPE: 17.55% | Time: 37.0s
Fold 3 | R2: 0.356 | RMSE: 38.39 | MAPE: 19.46% | Time: 42.2s
Fold 4 | R2: 0.218 | RMSE: 43.62 | MAPE: 22.92% | Time: 10.0s
Fold 5 | R2: 0.453 | RMSE: 35.71 | MAPE: 18.55% | Time: 16.7s

📊 FINAL SUMMARY: WEATHER
Total Time: 127.12s
-----------------------------------
R²:    0.3561 ± 0.0857
MAE:   29.15
RMSE:  38.32
NRMSE: 11.51%
MAPE:  19.70%
Acc:   0.5130
-----------------------------------


# SAR feature data set with weather and terrain features 

In [3]:
import os, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.neighbors import BallTree

# ---------------- 1. LOAD & CLEAN DATA ----------------
DATA_PATH = "data/corn_2016_2023_processed.parquet" 
df = pd.read_parquet(DATA_PATH)

# Ensure no NaNs in target
df = df[df["yield"].notna()].copy()
print(f"Loaded {len(df)} rows with {len(df.columns)} columns.")

# ---------------- 2. DYNAMIC FEATURE GROUPING ----------------
SAR_RAW = [c for c in df.columns if (c.startswith("VV_") or c.startswith("VH_")) and not c.endswith("_2")]
SAR_INDICES = [c for c in df.columns if c.startswith(("PR_", "RVI_"))]
WEATHER = [c for c in df.columns if any(k in c for k in ["prcp_", "srad_", "gdd_"]) and "_season" not in c]
TERRAIN = ["DEM", "slope_deg", "aspect_deg", "shape_index"]
SOIL    = [c for c in df.columns if c.startswith("soil_")]

SAR_ALL = SAR_RAW + SAR_INDICES

# ---------------- 3. SPATIAL FEATURE GENERATION ----------------
print("Calculating Field-Level Spatial Features (Means)...")
field_means = df.groupby(['farm_name', 'Year'])[SAR_ALL].transform('mean').astype(np.float32)
field_means.columns = [f"{c}_field_mean" for c in field_means.columns]
df = pd.concat([df, field_means], axis=1)

SAR_SPATIAL = SAR_ALL + list(field_means.columns)

def add_spatial_neighbor_features(df, sar_cols, n_neighbors=8):
    print(f"Generating BallTree neighbor features for {len(sar_cols)} features...")
    neighbor_data = np.zeros((len(df), len(sar_cols)), dtype=np.float32)
    
    for farm in df['farm_name'].unique():
        farm_mask = df['farm_name'] == farm
        farm_df = df[farm_mask]
        if len(farm_df) <= n_neighbors: continue
            
        tree = BallTree(farm_df[['latitude', 'longitude']].values)
        _, indices = tree.query(farm_df[['latitude', 'longitude']].values, k=n_neighbors+1)
        
        for i, col in enumerate(sar_cols):
            vals = farm_df[col].values
            neighbor_data[farm_mask, i] = vals[indices[:, 1:]].mean(axis=1)
            
    new_cols = [f"{c}_neighbor_8" for c in sar_cols]
    return pd.concat([df, pd.DataFrame(neighbor_data, columns=new_cols, index=df.index)], axis=1), new_cols

df, SAR_NEIGHBORS = add_spatial_neighbor_features(df, SAR_ALL)

# ---------------- 4. CONFIGURATION ----------------
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 50))
RUN_MODE = "TERRAIN" 

FEATURE_MAP = {
    "BASE":         SAR_ALL + ["Year", "latitude"],
    "WEATHER":      SAR_ALL + WEATHER + ["Year", "latitude"],
    "TERRAIN":      SAR_ALL + WEATHER + TERRAIN + SOIL + ["Year", "latitude"],
    "FULL_SPATIAL": SAR_SPATIAL + SAR_NEIGHBORS + WEATHER + TERRAIN + SOIL + ["Year", "latitude", "longitude"]
}

SELECTED_FEATURES = FEATURE_MAP[RUN_MODE]
X = df[SELECTED_FEATURES]
y = df["yield"].to_numpy(dtype=np.float32)
groups = df["farm_name"].astype(str).to_numpy()
years = df["Year"]

# Stratification for balanced folds
y_binned = pd.qcut(y, q=5, labels=False)

print(f"🚀 Mode: {RUN_MODE} | Features: {len(SELECTED_FEATURES)} | Device: H200 GPU")

# ---------------- 5. CV LOOP ----------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
overall_start = time.time()

for k, (tr, te) in enumerate(sgkf.split(X, y_binned, groups=groups), 1):
    f_start = time.time()
    
    X_tr, X_te = X.iloc[tr], X.iloc[te]
    y_tr, y_te = y[tr], y[te]
    
    # Year-Mean Normalization
    yr_map = df.iloc[tr].groupby("Year")["yield"].mean().to_dict()
    global_mean = y_tr.mean()
    mu_tr = years.iloc[tr].map(yr_map).fillna(global_mean).values
    mu_te = years.iloc[te].map(yr_map).fillna(global_mean).values

    # Model Definition (with Early Stopping in constructor for XGB 2.0+)
    model = xgb.XGBRegressor(
        tree_method="hist",
        device="cuda",
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.7,
        colsample_bytree=0.6,
        n_jobs=CPUS,
        random_state=42,
        reg_lambda=10.0,
        reg_alpha=1.0,
        early_stopping_rounds=50
    )
    
    model.fit(X_tr, y_tr / mu_tr, eval_set=[(X_te, y_te / mu_te)], verbose=False)
    
    p_ratio = model.predict(X_te)
    p_raw = p_ratio * mu_te

    # --- METRIC CALCULATIONS ---
    r2 = r2_score(y_te, p_raw)
    mae = mean_absolute_error(y_te, p_raw)
    rmse = np.sqrt(mean_squared_error(y_te, p_raw))
    
    y_range = np.max(y_te) - np.min(y_te)
    nrmse = (rmse / y_range) * 100 if y_range != 0 else 0
    mape = mean_absolute_percentage_error(y_te, p_raw) * 100
    
    z_true = np.where(y_te/mu_te < 0.9, 0, np.where(y_te/mu_te <= 1.1, 1, 2))
    z_pred = np.where(p_ratio < 0.9, 0, np.where(p_ratio <= 1.1, 1, 2))
    acc = (z_true == z_pred).mean()

    results.append({
        'R2': r2, 'MAE': mae, 'RMSE': rmse, 
        'NRMSE': nrmse, 'MAPE': mape, 'Zone_Acc': acc
    })

    print(f"Fold {k} | R2: {r2:.3f} | RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | Time: {time.time()-f_start:.1f}s")

# ---------------- 6. DETAILED SUMMARY ----------------
res_df = pd.DataFrame(results)

print(f"\n📊 FINAL SUMMARY: {RUN_MODE}")
print(f"Total Time: {time.time()-overall_start:.2f}s")
print("-" * 35)
print(f"R²:    {res_df['R2'].mean():.4f} ± {res_df['R2'].std():.4f}")
print(f"MAE:   {res_df['MAE'].mean():.2f}")
print(f"RMSE:  {res_df['RMSE'].mean():.2f}")
print(f"NRMSE: {res_df['NRMSE'].mean():.2f}%")
print(f"MAPE:  {res_df['MAPE'].mean():.2f}%")
print(f"Acc:   {res_df['Zone_Acc'].mean():.4f}")
print("-" * 35)


Loaded 2108996 rows with 323 columns.
Calculating Field-Level Spatial Features (Means)...
Generating BallTree neighbor features for 264 features...
🚀 Mode: TERRAIN | Features: 296 | Device: H200 GPU
Fold 1 | R2: 0.414 | RMSE: 36.78 | MAPE: 19.25% | Time: 8.7s
Fold 2 | R2: 0.449 | RMSE: 33.80 | MAPE: 16.37% | Time: 15.5s
Fold 3 | R2: 0.375 | RMSE: 37.80 | MAPE: 19.07% | Time: 13.5s
Fold 4 | R2: 0.229 | RMSE: 43.31 | MAPE: 22.86% | Time: 10.5s
Fold 5 | R2: 0.474 | RMSE: 35.04 | MAPE: 18.35% | Time: 16.5s

📊 FINAL SUMMARY: TERRAIN
Total Time: 67.14s
-----------------------------------
R²:    0.3882 ± 0.0963
MAE:   28.47
RMSE:  37.35
NRMSE: 11.23%
MAPE:  19.18%
Acc:   0.5222
-----------------------------------


# SAR features data set with weather, terrain features, and spatial feature

In [4]:
import os, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.neighbors import BallTree

# ---------------- 1. LOAD & CLEAN DATA ----------------
DATA_PATH = "data/corn_2016_2023_processed.parquet" 
df = pd.read_parquet(DATA_PATH)

# Ensure no NaNs in target
df = df[df["yield"].notna()].copy()
print(f"Loaded {len(df)} rows with {len(df.columns)} columns.")

# ---------------- 2. DYNAMIC FEATURE GROUPING ----------------
SAR_RAW = [c for c in df.columns if (c.startswith("VV_") or c.startswith("VH_")) and not c.endswith("_2")]
SAR_INDICES = [c for c in df.columns if c.startswith(("PR_", "RVI_"))]
WEATHER = [c for c in df.columns if any(k in c for k in ["prcp_", "srad_", "gdd_"]) and "_season" not in c]
TERRAIN = ["DEM", "slope_deg", "aspect_deg", "shape_index"]
SOIL    = [c for c in df.columns if c.startswith("soil_")]

SAR_ALL = SAR_RAW + SAR_INDICES

# ---------------- 3. SPATIAL FEATURE GENERATION ----------------
print("Calculating Field-Level Spatial Features (Means)...")
field_means = df.groupby(['farm_name', 'Year'])[SAR_ALL].transform('mean').astype(np.float32)
field_means.columns = [f"{c}_field_mean" for c in field_means.columns]
df = pd.concat([df, field_means], axis=1)

SAR_SPATIAL = SAR_ALL + list(field_means.columns)

def add_spatial_neighbor_features(df, sar_cols, n_neighbors=8):
    print(f"Generating BallTree neighbor features for {len(sar_cols)} features...")
    neighbor_data = np.zeros((len(df), len(sar_cols)), dtype=np.float32)
    
    for farm in df['farm_name'].unique():
        farm_mask = df['farm_name'] == farm
        farm_df = df[farm_mask]
        if len(farm_df) <= n_neighbors: continue
            
        tree = BallTree(farm_df[['latitude', 'longitude']].values)
        _, indices = tree.query(farm_df[['latitude', 'longitude']].values, k=n_neighbors+1)
        
        for i, col in enumerate(sar_cols):
            vals = farm_df[col].values
            neighbor_data[farm_mask, i] = vals[indices[:, 1:]].mean(axis=1)
            
    new_cols = [f"{c}_neighbor_8" for c in sar_cols]
    return pd.concat([df, pd.DataFrame(neighbor_data, columns=new_cols, index=df.index)], axis=1), new_cols

df, SAR_NEIGHBORS = add_spatial_neighbor_features(df, SAR_ALL)

# ---------------- 4. CONFIGURATION ----------------
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 50))
RUN_MODE = "FULL_SPATIAL" 

FEATURE_MAP = {
    "BASE":         SAR_ALL + ["Year", "latitude"],
    "WEATHER":      SAR_ALL + WEATHER + ["Year", "latitude"],
    "TERRAIN":      SAR_ALL + WEATHER + TERRAIN + SOIL + ["Year", "latitude"],
    "FULL_SPATIAL": SAR_SPATIAL + SAR_NEIGHBORS + WEATHER + TERRAIN + SOIL + ["Year", "latitude", "longitude"]
}

SELECTED_FEATURES = FEATURE_MAP[RUN_MODE]
X = df[SELECTED_FEATURES]
y = df["yield"].to_numpy(dtype=np.float32)
groups = df["farm_name"].astype(str).to_numpy()
years = df["Year"]

# Stratification for balanced folds
y_binned = pd.qcut(y, q=5, labels=False)

print(f"🚀 Mode: {RUN_MODE} | Features: {len(SELECTED_FEATURES)} | Device: H200 GPU")

# ---------------- 5. CV LOOP ----------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
overall_start = time.time()

for k, (tr, te) in enumerate(sgkf.split(X, y_binned, groups=groups), 1):
    f_start = time.time()
    
    X_tr, X_te = X.iloc[tr], X.iloc[te]
    y_tr, y_te = y[tr], y[te]
    
    # Year-Mean Normalization
    yr_map = df.iloc[tr].groupby("Year")["yield"].mean().to_dict()
    global_mean = y_tr.mean()
    mu_tr = years.iloc[tr].map(yr_map).fillna(global_mean).values
    mu_te = years.iloc[te].map(yr_map).fillna(global_mean).values

    # Model Definition (with Early Stopping in constructor for XGB 2.0+)
    model = xgb.XGBRegressor(
        tree_method="hist",
        device="cuda",
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.7,
        colsample_bytree=0.6,
        n_jobs=CPUS,
        random_state=42,
        reg_lambda=10.0,
        reg_alpha=1.0,
        early_stopping_rounds=50
    )
    
    model.fit(X_tr, y_tr / mu_tr, eval_set=[(X_te, y_te / mu_te)], verbose=False)
    
    p_ratio = model.predict(X_te)
    p_raw = p_ratio * mu_te

    # --- METRIC CALCULATIONS ---
    r2 = r2_score(y_te, p_raw)
    mae = mean_absolute_error(y_te, p_raw)
    rmse = np.sqrt(mean_squared_error(y_te, p_raw))
    
    y_range = np.max(y_te) - np.min(y_te)
    nrmse = (rmse / y_range) * 100 if y_range != 0 else 0
    mape = mean_absolute_percentage_error(y_te, p_raw) * 100
    
    z_true = np.where(y_te/mu_te < 0.9, 0, np.where(y_te/mu_te <= 1.1, 1, 2))
    z_pred = np.where(p_ratio < 0.9, 0, np.where(p_ratio <= 1.1, 1, 2))
    acc = (z_true == z_pred).mean()

    results.append({
        'R2': r2, 'MAE': mae, 'RMSE': rmse, 
        'NRMSE': nrmse, 'MAPE': mape, 'Zone_Acc': acc
    })

    print(f"Fold {k} | R2: {r2:.3f} | RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | Time: {time.time()-f_start:.1f}s")

# ---------------- 6. DETAILED SUMMARY ----------------
res_df = pd.DataFrame(results)

print(f"\n📊 FINAL SUMMARY: {RUN_MODE}")
print(f"Total Time: {time.time()-overall_start:.2f}s")
print("-" * 35)
print(f"R²:    {res_df['R2'].mean():.4f} ± {res_df['R2'].std():.4f}")
print(f"MAE:   {res_df['MAE'].mean():.2f}")
print(f"RMSE:  {res_df['RMSE'].mean():.2f}")
print(f"NRMSE: {res_df['NRMSE'].mean():.2f}%")
print(f"MAPE:  {res_df['MAPE'].mean():.2f}%")
print(f"Acc:   {res_df['Zone_Acc'].mean():.4f}")
print("-" * 35)


Loaded 2108996 rows with 323 columns.
Calculating Field-Level Spatial Features (Means)...
Generating BallTree neighbor features for 264 features...
🚀 Mode: FULL_SPATIAL | Features: 825 | Device: H200 GPU
Fold 1 | R2: 0.436 | RMSE: 36.10 | MAPE: 18.52% | Time: 16.4s
Fold 2 | R2: 0.463 | RMSE: 33.37 | MAPE: 16.08% | Time: 21.8s
Fold 3 | R2: 0.384 | RMSE: 37.53 | MAPE: 18.92% | Time: 17.1s
Fold 4 | R2: 0.276 | RMSE: 41.96 | MAPE: 21.92% | Time: 12.7s
Fold 5 | R2: 0.491 | RMSE: 34.45 | MAPE: 18.11% | Time: 30.5s

📊 FINAL SUMMARY: FULL_SPATIAL
Total Time: 100.92s
-----------------------------------
R²:    0.4101 ± 0.0845
MAE:   27.89
RMSE:  36.68
NRMSE: 11.03%
MAPE:  18.71%
Acc:   0.5331
-----------------------------------


# SAR features data set with weather, terrain features, spatial features, and ablations 

In [1]:
# =======================
# FULL MEMORY RESET HEADER
# =======================

import gc, os

# limit thread pools
os.environ["OMP_NUM_THREADS"] = "50"
os.environ["OPENBLAS_NUM_THREADS"] = "50"
os.environ["MKL_NUM_THREADS"] = "50"
os.environ["NUMEXPR_NUM_THREADS"] = "50"

print("Clearing Python memory...")
for name in list(globals().keys()):
    if not name.startswith("_") and name not in ("gc", "os"):
        del globals()[name]
gc.collect()

# clear GPU memory
try:
    import numba.cuda as cuda
    cuda.select_device(0)
    cuda.close()
    print("GPU memory cleared.")
except Exception:
    pass

print("Memory reset complete.")

import os, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.neighbors import BallTree

# ---------------- 1. LOAD & CLEAN DATA ----------------
ORIGINAL_PATH = "data/corn_2016_2023_processed.parquet"
ABLATED_PATH = "data/gamma_k8_stacked_ablated_corn_wheat_soy.parquet"

df_orig = pd.read_parquet(ORIGINAL_PATH)
df_orig.columns = [c.lower() for c in df_orig.columns]
df_orig = df_orig[df_orig["yield"].notna()].copy()
df_orig['is_ablation'] = False

valid_years = df_orig['year'].unique()

# Load and Filter Ablated Data
df_ablated = pd.read_parquet(ABLATED_PATH)
df_ablated.columns = [c.lower() for c in df_ablated.columns]

# FILTER: Soy only + Matching Years only
df_ablated = df_ablated[
    (df_ablated['crop'].str.contains('corn', case=False, na=False)) & 
    (df_ablated['year'].isin(valid_years))
].copy()
df_ablated['is_ablation'] = True

# Rename Ablated Farms to avoid leakage in GroupKFold
unique_farms = df_ablated['farm_name'].unique()
farm_id_map = {name: f"ablation_field_ID_{i}" for i, name in enumerate(unique_farms)}
df_ablated['farm_name'] = df_ablated['farm_name'].map(farm_id_map)

# Combine datasets
common_cols = list(set(df_orig.columns) & set(df_ablated.columns))
df = pd.concat([df_orig[common_cols + ['is_ablation']], 
                df_ablated[common_cols + ['is_ablation']]], axis=0).reset_index(drop=True)

print(f"Loaded {len(df_orig)} real rows and {len(df_ablated)} filtered ablated rows.")

# ---------------- 2. FEATURE GROUPS & SPATIAL ----------------
SAR_RAW = [c for c in df.columns if (c.startswith("vv_") or c.startswith("vh_")) and not c.endswith("_2")]
SAR_INDICES = [c for c in df.columns if c.startswith(("pr_", "rvi_"))]
WEATHER = [c for c in df.columns if any(k in c for k in ["prcp_", "srad_", "gdd_"]) and "_season" not in c]
TERRAIN = ["dem", "slope_deg", "aspect_deg", "shape_index"]
SOIL    = [c for c in df.columns if c.startswith("soil_")]
SAR_ALL = SAR_RAW + SAR_INDICES

print("Calculating Spatial Features...")
field_means = df.groupby(['farm_name', 'year'])[SAR_ALL].transform('mean').astype(np.float32)
field_means.columns = [f"{c}_field_mean" for c in field_means.columns]
df = pd.concat([df, field_means], axis=1)

def add_spatial_neighbors(df, sar_cols, n=8):
    print(f"Generating BallTree neighbor features for {len(sar_cols)} features...")
    neighbor_data = np.zeros((len(df), len(sar_cols)), dtype=np.float32)
    for farm in df['farm_name'].unique():
        mask = df['farm_name'] == farm
        f_df = df[mask]
        if len(f_df) <= n: continue
        tree = BallTree(f_df[['latitude', 'longitude']].values)
        _, idx = tree.query(f_df[['latitude', 'longitude']].values, k=n+1)
        for i, col in enumerate(sar_cols):
            neighbor_data[mask, i] = f_df[col].values[idx[:, 1:]].mean(axis=1)
    new_cols = [f"{c}_neighbor_8" for c in sar_cols]
    return pd.concat([df, pd.DataFrame(neighbor_data, columns=new_cols, index=df.index)], axis=1), new_cols

df, SAR_NEIGHBORS = add_spatial_neighbors(df, SAR_ALL)

# ---------------- 3. CONFIGURATION ----------------
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 50))
RUN_MODE = "FULL_SPATIAL_WITH_ABLATION"

SELECTED_FEATURES = SAR_ALL + list(field_means.columns) + SAR_NEIGHBORS + WEATHER + TERRAIN + SOIL + ["year", "latitude"]
X = df[SELECTED_FEATURES]
y = df["yield"].to_numpy(dtype=np.float32)
groups = df["farm_name"].astype(str).to_numpy()
years = df["year"].to_numpy()
is_ablation = df["is_ablation"].to_numpy().flatten()

y_binned = pd.qcut(y, q=5, labels=False)
print(f"🚀 Mode: {RUN_MODE} | Features: {len(SELECTED_FEATURES)} | Device: GPU")

# ---------------- 4. CV LOOP ----------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
overall_start = time.time()

for k, (tr, te) in enumerate(sgkf.split(X, y_binned, groups=groups), 1):
    f_start = time.time()
    
    # Filter test set for REAL fields only
    te_bool_mask = (is_ablation[te] == False)
    real_te_idx = te[te_bool_mask]
    
    if len(real_te_idx) == 0: continue
    
    X_tr, X_te_real = X.iloc[tr], X.iloc[real_te_idx]
    y_te_real = y[real_te_idx]
    
    # Normalization factors
    yr_map = df.iloc[tr].groupby("year")["yield"].mean().to_dict()
    global_mean = y[tr].mean()
    mu_tr = pd.Series(years[tr]).map(yr_map).fillna(global_mean).values
    mu_te_real = pd.Series(years[real_te_idx]).map(yr_map).fillna(global_mean).values

    model = xgb.XGBRegressor(
        tree_method="hist", device="cuda", n_estimators=1000, max_depth=6,
        learning_rate=0.03, subsample=0.7, colsample_bytree=0.6,
        n_jobs=CPUS, random_state=42, reg_lambda=10.0, early_stopping_rounds=50
    )
    
    model.fit(X_tr, y[tr] / mu_tr, eval_set=[(X_te_real, y_te_real / mu_te_real)], verbose=False)
    
    p_ratio = model.predict(X_te_real)
    p_raw = p_ratio * mu_te_real

    # Metrics
    r2 = r2_score(y_te_real, p_raw)
    mae = mean_absolute_error(y_te_real, p_raw)
    rmse = np.sqrt(mean_squared_error(y_te_real, p_raw))
    mape = mean_absolute_percentage_error(y_te_real, p_raw) * 100
    y_range = np.max(y_te_real) - np.min(y_te_real)
    nrmse = (rmse / y_range) * 100 if y_range != 0 else 0
    
    z_true = np.where(y_te_real/mu_te_real < 0.9, 0, np.where(y_te_real/mu_te_real <= 1.1, 1, 2))
    z_pred = np.where(p_ratio < 0.9, 0, np.where(p_ratio <= 1.1, 1, 2))
    acc = (z_true == z_pred).mean()

    results.append({'R2': r2, 'MAE': mae, 'RMSE': rmse, 'NRMSE': nrmse, 'MAPE': mape, 'Zone_Acc': acc})
    print(f"Fold {k} | R2: {r2:.3f} | RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | Time: {time.time()-f_start:.1f}s")

# ---------------- 5. FINAL SUMMARY ----------------
res_df = pd.DataFrame(results)
print(f"\n📊 FINAL SUMMARY: {RUN_MODE}")
print(f"Total Time: {time.time()-overall_start:.2f}s")
print("-" * 35)
print(f"R²:    {res_df['R2'].mean():.4f} ± {res_df['R2'].std():.4f}")
print(f"MAE:   {res_df['MAE'].mean():.2f}")
print(f"RMSE:  {res_df['RMSE'].mean():.2f}")
print(f"NRMSE: {res_df['NRMSE'].mean():.2f}%")
print(f"MAPE:  {res_df['MAPE'].mean():.2f}%")
print(f"Acc:   {res_df['Zone_Acc'].mean():.4f}")
print("-" * 35)

Clearing Python memory...
Memory reset complete.
Loaded 2108996 real rows and 3207789 filtered ablated rows.
Calculating Spatial Features...
Generating BallTree neighbor features for 132 features...
🚀 Mode: FULL_SPATIAL_WITH_ABLATION | Features: 428 | Device: GPU
Fold 1 | R2: 0.592 | RMSE: 30.08 | MAPE: 15.52% | Time: 43.4s
Fold 2 | R2: 0.605 | RMSE: 29.33 | MAPE: 14.77% | Time: 53.0s
Fold 3 | R2: 0.587 | RMSE: 30.64 | MAPE: 16.84% | Time: 51.5s
Fold 4 | R2: 0.553 | RMSE: 31.72 | MAPE: 16.18% | Time: 54.2s
Fold 5 | R2: 0.586 | RMSE: 30.35 | MAPE: 15.25% | Time: 140.8s

📊 FINAL SUMMARY: FULL_SPATIAL_WITH_ABLATION
Total Time: 349.65s
-----------------------------------
R²:    0.5844 ± 0.0191
MAE:   22.40
RMSE:  30.43
NRMSE: 8.89%
MAPE:  15.71%
Acc:   0.6372
-----------------------------------


# SAR features data set with weather, terrain features, spatial features, and ALL ablations

In [1]:
# =======================
# FULL MEMORY RESET HEADER
# =======================

import gc, os

# limit thread pools
os.environ["OMP_NUM_THREADS"] = "50"
os.environ["OPENBLAS_NUM_THREADS"] = "50"
os.environ["MKL_NUM_THREADS"] = "50"
os.environ["NUMEXPR_NUM_THREADS"] = "50"

print("Clearing Python memory...")
for name in list(globals().keys()):
    if not name.startswith("_") and name not in ("gc", "os"):
        del globals()[name]
gc.collect()

# clear GPU memory
try:
    import numba.cuda as cuda
    cuda.select_device(0)
    cuda.close()
    print("GPU memory cleared.")
except Exception:
    pass

print("Memory reset complete.")

import os, time
import numpy as np
import torch
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.neighbors import BallTree

# ---------------- 1. LOAD & CLEAN DATA ----------------
ORIGINAL_PATH = "data/corn_2016_2023_processed.parquet"

# NEW ablated file (SAR + TERRAIN + WEATHER ablated)
ABLATED_PATH = "data/gamma_k8_stacked_ablated_TERRAIN_WEATHER_corn_wheat_soy.parquet"

df_orig = pd.read_parquet(ORIGINAL_PATH)
df_orig.columns = [c.lower() for c in df_orig.columns]
df_orig = df_orig[df_orig["yield"].notna()].copy()
df_orig["is_ablation"] = False

valid_years = df_orig["year"].unique()

df_ablated = pd.read_parquet(ABLATED_PATH)
df_ablated.columns = [c.lower() for c in df_ablated.columns]

# FILTER: Soy only + Matching Years only
df_ablated = df_ablated[
    (df_ablated["crop"].str.contains("corn", case=False, na=False)) &
    (df_ablated["year"].isin(valid_years))
].copy()
df_ablated["is_ablation"] = True

# Rename Ablated Farms to avoid leakage in GroupKFold
unique_farms = df_ablated["farm_name"].unique()
farm_id_map = {name: f"ablation_field_id_{i}" for i, name in enumerate(unique_farms)}
df_ablated["farm_name"] = df_ablated["farm_name"].map(farm_id_map)

# ---- IMPORTANT CHANGE: use UNION of columns, not intersection ----
# Keep yield in both, and align schemas by reindexing.
all_cols = sorted(set(df_orig.columns) | set(df_ablated.columns))
df_orig_aligned = df_orig.reindex(columns=all_cols)
df_ablated_aligned = df_ablated.reindex(columns=all_cols)

df = pd.concat([df_orig_aligned, df_ablated_aligned], axis=0, ignore_index=True)
print(f"Loaded {len(df_orig)} real rows and {len(df_ablated)} filtered ablated rows.")

# ---------------- 2. FEATURE GROUPS & SPATIAL ----------------
# SAR (as before)
SAR_RAW = [c for c in df.columns if (c.startswith("vv_") or c.startswith("vh_")) and not c.endswith("_2")]
SAR_INDICES = [c for c in df.columns if c.startswith(("pr_", "rvi_"))]
SAR_ALL = SAR_RAW + SAR_INDICES

# ---- IMPORTANT CHANGE: include *_season weather too ----
WEATHER = [c for c in df.columns if any(k in c for k in ["prcp_", "srad_", "gdd_"])]
# (this will include prcp_04..prcp_10 AND prcp_season, etc.)

TERRAIN = ["dem", "slope_deg", "aspect_deg", "shape_index"]
SOIL = [c for c in df.columns if c.startswith("soil_")]

# Only compute spatial features from SAR (keeps your design intact)
print("Calculating Spatial Features...")
field_means = df.groupby(["farm_name", "year"])[SAR_ALL].transform("mean").astype(np.float32)
field_means.columns = [f"{c}_field_mean" for c in field_means.columns]
df = pd.concat([df, field_means], axis=1)

def add_spatial_neighbors_fast(df, cols, n=8, only_real=True):
    """
    Vectorized BallTree neighbor mean features.
    - Computes per-farm neighbor means for ALL columns at once.
    - Optionally computes ONLY for real rows (recommended).
    """
    print(f"Generating BallTree neighbor features for {len(cols)} features... (n={n}, only_real={only_real})")

    # Output array (float32, NaN default)
    out = np.full((len(df), len(cols)), np.nan, dtype=np.float32)

    # We'll optionally skip ablated rows to save huge time
    if only_real and "is_ablation" in df.columns:
        base_mask = ~df["is_ablation"].astype(bool)
    else:
        base_mask = pd.Series(True, index=df.index)

    # Iterate farms (still needed for separate trees), but vectorized inside
    for farm, idx in df.loc[base_mask].groupby("farm_name").groups.items():
        idx = np.array(list(idx), dtype=int)
        if idx.size <= n:
            continue

        f_df = df.loc[idx]
        coords = f_df[["latitude", "longitude"]].to_numpy(dtype=np.float64)

        # Build tree and query neighbors
        tree = BallTree(coords)
        _, nn = tree.query(coords, k=n+1)       # nn shape: (m, n+1), includes self
        nn = nn[:, 1:]                          # drop self -> (m, n)

        # Feature matrix (m, p)
        A = f_df[cols].to_numpy(dtype=np.float32)

        # Gather neighbors: (m, n, p)
        neigh = A[nn]  # uses fancy indexing

        # NaN-safe mean along neighbor axis
        s = np.nansum(neigh, axis=1)  # (m, p)
        c = np.sum(~np.isnan(neigh), axis=1)  # (m, p)
        m = s / np.maximum(c, 1)  # avoid division by 0
        m[c == 0] = np.nan        # where all neighbors are NaN -> keep NaN (no warning)

        out[idx, :] = m

    new_cols = [f"{c}_neighbor_{n}" for c in cols]
    df = pd.concat([df, pd.DataFrame(out, columns=new_cols, index=df.index)], axis=1)
    return df, new_cols


df, SAR_NEIGHBORS = add_spatial_neighbors_fast(df, SAR_ALL, n=8, only_real=True)


# ---------------- 3. CONFIGURATION ----------------
import gc # Added for manual memory clearing

# ---------------- 3. CONFIGURATION (Memory Optimized) ----------------
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 50))
RUN_MODE = "FULL_SPATIAL_WITH_ABLATED_SAR_TERRAIN_WEATHER"

SELECTED_FEATURES = (
    SAR_ALL
    + list(field_means.columns)
    + SAR_NEIGHBORS
    + WEATHER
    + TERRAIN
    + SOIL
    + ["year", "latitude"]
)
SELECTED_FEATURES = [c for c in SELECTED_FEATURES if c in df.columns]

# --- CRITICAL: Force float32 to save 50% RAM compared to float64 ---
X = df[SELECTED_FEATURES].astype(np.float32)
y = df["yield"].to_numpy(dtype=np.float32)
groups = df["farm_name"].astype(str).to_numpy()
years = df["year"].to_numpy(dtype=np.int32)
is_ablation = df["is_ablation"].to_numpy().flatten()

# Clear the original massive dataframe to free up ~30GB
del df
del field_means
gc.collect() 

y_binned = pd.qcut(y, q=5, labels=False, duplicates="drop")
print(f"🚀 Mode: {RUN_MODE} | Features: {len(SELECTED_FEATURES)} | Device: GPU")

# ---------------- 4. CV LOOP (FULL METRICS - NO TRIMMING) ----------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
overall_start = time.time()

for k, (tr, te) in enumerate(sgkf.split(X, y_binned, groups=groups), 1):
    f_start = time.time()

    # Test only on REAL fields
    te_bool_mask = (is_ablation[te] == False)
    real_te_idx = te[te_bool_mask]
    if len(real_te_idx) == 0: continue

    X_tr, X_te_real = X.iloc[tr], X.iloc[real_te_idx]
    y_tr, y_te_real = y[tr], y[real_te_idx]

    # Normalization factors
    unique_yrs = np.unique(years[tr])
    yr_map = {yr: y_tr[years[tr] == yr].mean() for yr in unique_yrs}
    global_mean = y_tr.mean()
    
    mu_tr = np.array([yr_map.get(yr, global_mean) for yr in years[tr]])
    mu_te_real = np.array([yr_map.get(yr, global_mean) for yr in years[real_te_idx]])

    model = xgb.XGBRegressor(
        tree_method="hist",
        device="cuda",
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.7,
        colsample_bytree=0.6,
        n_jobs=CPUS,
        random_state=42,
        reg_lambda=10.0,
        early_stopping_rounds=50,
    )

    model.fit(
        X_tr,
        y_tr / mu_tr,
        eval_set=[(X_te_real, y_te_real / mu_te_real)],
        verbose=False
    )

    p_ratio = model.predict(X_te_real)
    p_raw = p_ratio * mu_te_real

    # --- ALL METRICS RESTORED ---
    r2 = r2_score(y_te_real, p_raw)
    mae = mean_absolute_error(y_te_real, p_raw)
    rmse = np.sqrt(mean_squared_error(y_te_real, p_raw))
    mape = mean_absolute_percentage_error(y_te_real, p_raw) * 100
    
    y_range = np.max(y_te_real) - np.min(y_te_real)
    nrmse = (rmse / y_range) * 100 if y_range != 0 else 0

    # Zone Accuracy logic (0.9 - 1.1 threshold)
    z_true = np.where(y_te_real/mu_te_real < 0.9, 0, np.where(y_te_real/mu_te_real <= 1.1, 1, 2))
    z_pred = np.where(p_ratio < 0.9, 0, np.where(p_ratio <= 1.1, 1, 2))
    acc = (z_true == z_pred).mean()

    results.append({
        "R2": r2, 
        "MAE": mae, 
        "RMSE": rmse, 
        "NRMSE": nrmse, 
        "MAPE": mape, 
        "Zone_Acc": acc
    })
    
    print(f"Fold {k} | R2: {r2:.3f} | MAE: {mae:.2f} | RMSE: {rmse:.2f} | Acc: {acc:.4f}")
    
    # Cleanup to prevent crash
    del X_tr, X_te_real, model
    gc.collect() 
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ---------------- 5. FINAL SUMMARY ----------------
res_df = pd.DataFrame(results)
print(f"\n📊 FINAL SUMMARY: {RUN_MODE}")
print("-" * 35)
print(f"R²:       {res_df['R2'].mean():.4f} ± {res_df['R2'].std():.4f}")
print(f"MAE:      {res_df['MAE'].mean():.2f}")
print(f"RMSE:     {res_df['RMSE'].mean():.2f}")
print(f"NRMSE:    {res_df['NRMSE'].mean():.2f}%")
print(f"MAPE:     {res_df['MAPE'].mean():.2f}%")
print(f"Zone Acc: {res_df['Zone_Acc'].mean():.4f}")
print("-" * 35)

Clearing Python memory...
Memory reset complete.
Loaded 2108996 real rows and 3207789 filtered ablated rows.
Calculating Spatial Features...
Generating BallTree neighbor features for 692 features... (n=8, only_real=True)
🚀 Mode: FULL_SPATIAL_WITH_ABLATED_SAR_TERRAIN_WEATHER | Features: 2111 | Device: GPU
Fold 1 | R2: 0.497 | MAE: 26.33 | RMSE: 34.72 | Acc: 0.5774
Fold 2 | R2: 0.595 | MAE: 22.94 | RMSE: 30.35 | Acc: 0.6206
Fold 3 | R2: 0.613 | MAE: 21.36 | RMSE: 29.49 | Acc: 0.6583
Fold 4 | R2: 0.537 | MAE: 24.00 | RMSE: 33.41 | Acc: 0.6390
Fold 5 | R2: 0.529 | MAE: 23.55 | RMSE: 31.48 | Acc: 0.6041

📊 FINAL SUMMARY: FULL_SPATIAL_WITH_ABLATED_SAR_TERRAIN_WEATHER
-----------------------------------
R²:       0.5542 ± 0.0483
MAE:      23.64
RMSE:     31.89
NRMSE:    9.53%
MAPE:     15.93%
Zone Acc: 0.6199
-----------------------------------


# Making Prediction Column

In [1]:
import os, time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.neighbors import BallTree

# ---------------- 1. LOAD & CLEAN DATA ----------------
ORIGINAL_PATH = "data/corn_2016_2023_processed.parquet"
ABLATED_PATH  = "data/gamma_k8_stacked_ablated_corn_wheat_soy.parquet"

OUTPUT_PATH   = "data/corn_full_spatial_with_ablation__with_oof_preds.parquet"
PRED_COL      = "pred_yield_oof"
PRED_FOLD_COL = "pred_fold"

TARGET_CROP = "corn"  # ✅ IMPORTANT

df_orig = pd.read_parquet(ORIGINAL_PATH)
df_orig.columns = [c.lower() for c in df_orig.columns]
df_orig = df_orig[df_orig["yield"].notna()].copy()
df_orig["is_ablation"] = False

valid_years = df_orig["year"].unique()

# Load and Filter Ablated Data
df_ablated = pd.read_parquet(ABLATED_PATH)
df_ablated.columns = [c.lower() for c in df_ablated.columns]

# ✅ FILTER: CORN only + Matching Years only
df_ablated = df_ablated[
    (df_ablated["crop"].str.contains(TARGET_CROP, case=False, na=False)) &
    (df_ablated["year"].isin(valid_years))
].copy()
df_ablated["is_ablation"] = True

# Rename Ablated Farms to avoid leakage in GroupKFold
unique_farms = df_ablated["farm_name"].unique()
farm_id_map = {name: f"ablation_field_ID_{i}" for i, name in enumerate(unique_farms)}
df_ablated["farm_name"] = df_ablated["farm_name"].map(farm_id_map)

# Combine datasets (deterministic column order + no duplicate is_ablation)
common_cols = [c for c in df_orig.columns if c in df_ablated.columns]
# Ensure is_ablation appears exactly once and at the end (nice for readability)
common_cols = [c for c in common_cols if c != "is_ablation"] + ["is_ablation"]

df = pd.concat(
    [df_orig[common_cols], df_ablated[common_cols]],
    axis=0
).reset_index(drop=True)

# Safety: drop any accidental duplicates (should be none now)
df = df.loc[:, ~df.columns.duplicated()]

print(f"Loaded {len(df_orig)} real rows and {len(df_ablated)} filtered ablated rows for crop='{TARGET_CROP}'.")

# ---------------- 2. FEATURE GROUPS & SPATIAL ----------------
SAR_RAW = [c for c in df.columns if (c.startswith("vv_") or c.startswith("vh_")) and not c.endswith("_2")]
SAR_INDICES = [c for c in df.columns if c.startswith(("pr_", "rvi_"))]
WEATHER = [c for c in df.columns if any(k in c for k in ["prcp_", "srad_", "gdd_"]) and "_season" not in c]
TERRAIN = ["dem", "slope_deg", "aspect_deg", "shape_index"]
SOIL    = [c for c in df.columns if c.startswith("soil_")]
SAR_ALL = SAR_RAW + SAR_INDICES

print("Calculating Spatial Features...")
field_means = df.groupby(["farm_name", "year"])[SAR_ALL].transform("mean").astype(np.float32)
field_means.columns = [f"{c}_field_mean" for c in field_means.columns]
df = pd.concat([df, field_means], axis=1)

def add_spatial_neighbors(df_in, sar_cols, n=8):
    print(f"Generating BallTree neighbor features for {len(sar_cols)} features...")
    neighbor_data = np.zeros((len(df_in), len(sar_cols)), dtype=np.float32)

    for farm in df_in["farm_name"].unique():
        mask = (df_in["farm_name"] == farm).to_numpy()
        f_df = df_in.loc[mask, :]
        if len(f_df) <= n:
            continue

        coords = f_df[["latitude", "longitude"]].to_numpy()
        tree = BallTree(coords)
        _, idx = tree.query(coords, k=n + 1)  # includes self at idx[:,0]

        for i, col in enumerate(sar_cols):
            vals = f_df[col].to_numpy()
            neighbor_data[mask, i] = vals[idx[:, 1:]].mean(axis=1)

    new_cols = [f"{c}_neighbor_{n}" for c in sar_cols]
    df_out = pd.concat([df_in, pd.DataFrame(neighbor_data, columns=new_cols, index=df_in.index)], axis=1)
    return df_out, new_cols

df, SAR_NEIGHBORS = add_spatial_neighbors(df, SAR_ALL, n=8)

# ---------------- 3. CONFIGURATION ----------------
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", 50))
RUN_MODE = "FULL_SPATIAL_WITH_ABLATION"

SELECTED_FEATURES = SAR_ALL + list(field_means.columns) + SAR_NEIGHBORS + WEATHER + TERRAIN + SOIL + ["year", "latitude"]

X = df[SELECTED_FEATURES]
y = df["yield"].to_numpy(dtype=np.float32)
groups = df["farm_name"].astype(str).to_numpy()
years = df["year"].to_numpy()
is_ablation = df["is_ablation"].to_numpy().flatten()

y_binned = pd.qcut(y, q=5, labels=False)
print(f"🚀 Mode: {RUN_MODE} | Features: {len(SELECTED_FEATURES)} | Device: GPU")

# Allocate prediction columns
df[PRED_COL] = np.nan
df[PRED_FOLD_COL] = np.nan

# ---------------- 4. CV LOOP ----------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
results = []
overall_start = time.time()

for k, (tr, te) in enumerate(sgkf.split(X, y_binned, groups=groups), 1):
    f_start = time.time()

    # Filter test set for REAL fields only
    te_bool_mask = (is_ablation[te] == False)
    real_te_idx = te[te_bool_mask]
    if len(real_te_idx) == 0:
        continue

    X_tr, X_te_real = X.iloc[tr], X.iloc[real_te_idx]
    y_te_real = y[real_te_idx]

    # Normalization factors
    yr_map = df.iloc[tr].groupby("year")["yield"].mean().to_dict()
    global_mean = float(y[tr].mean())
    mu_tr = pd.Series(years[tr]).map(yr_map).fillna(global_mean).values
    mu_te_real = pd.Series(years[real_te_idx]).map(yr_map).fillna(global_mean).values

    model = xgb.XGBRegressor(
        tree_method="hist", device="cuda", n_estimators=1000, max_depth=6,
        learning_rate=0.03, subsample=0.7, colsample_bytree=0.6,
        n_jobs=CPUS, random_state=42, reg_lambda=10.0, early_stopping_rounds=50
    )

    model.fit(X_tr, y[tr] / mu_tr, eval_set=[(X_te_real, y_te_real / mu_te_real)], verbose=False)

    p_ratio = model.predict(X_te_real)
    p_raw = p_ratio * mu_te_real

    # Store OOF predictions
    df.loc[real_te_idx, PRED_COL] = p_raw.astype(np.float32)
    df.loc[real_te_idx, PRED_FOLD_COL] = k

    # Metrics
    r2 = r2_score(y_te_real, p_raw)
    mae = mean_absolute_error(y_te_real, p_raw)
    rmse = np.sqrt(mean_squared_error(y_te_real, p_raw))
    mape = mean_absolute_percentage_error(y_te_real, p_raw) * 100
    y_range = np.max(y_te_real) - np.min(y_te_real)
    nrmse = (rmse / y_range) * 100 if y_range != 0 else 0

    z_true = np.where(y_te_real/mu_te_real < 0.9, 0, np.where(y_te_real/mu_te_real <= 1.1, 1, 2))
    z_pred = np.where(p_ratio < 0.9, 0, np.where(p_ratio <= 1.1, 1, 2))
    acc = (z_true == z_pred).mean()

    results.append({'R2': r2, 'MAE': mae, 'RMSE': rmse, 'NRMSE': nrmse, 'MAPE': mape, 'Zone_Acc': acc})
    print(f"Fold {k} | R2: {r2:.3f} | RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | Time: {time.time()-f_start:.1f}s")

# ---------------- 5. FINAL SUMMARY ----------------
res_df = pd.DataFrame(results)
print(f"\n📊 FINAL SUMMARY: {RUN_MODE}")
print(f"Total Time: {time.time()-overall_start:.2f}s")
print("-" * 35)
print(f"R²:    {res_df['R2'].mean():.4f} ± {res_df['R2'].std():.4f}")
print(f"MAE:   {res_df['MAE'].mean():.2f}")
print(f"RMSE:  {res_df['RMSE'].mean():.2f}")
print(f"NRMSE: {res_df['NRMSE'].mean():.2f}%")
print(f"MAPE:  {res_df['MAPE'].mean():.2f}%")
print(f"Acc:   {res_df['Zone_Acc'].mean():.4f}")
print("-" * 35)

# ---------------- 6. SAVE DATASET WITH PREDICTIONS ----------------
df = df.loc[:, ~df.columns.duplicated()]
df.to_parquet(OUTPUT_PATH, index=False)
print(f"\n✅ Saved dataframe (with OOF predictions) to: {OUTPUT_PATH}")
print(f"Columns added: {PRED_COL}, {PRED_FOLD_COL}")

Loaded 2108996 real rows and 3207789 filtered ablated rows for crop='corn'.
Calculating Spatial Features...
Generating BallTree neighbor features for 132 features...
🚀 Mode: FULL_SPATIAL_WITH_ABLATION | Features: 428 | Device: GPU
Fold 1 | R2: 0.584 | RMSE: 31.58 | MAPE: 15.82% | Time: 55.9s
Fold 2 | R2: 0.631 | RMSE: 28.96 | MAPE: 14.02% | Time: 37.0s
Fold 3 | R2: 0.613 | RMSE: 29.49 | MAPE: 15.74% | Time: 80.6s
Fold 4 | R2: 0.562 | RMSE: 32.47 | MAPE: 16.47% | Time: 91.7s
Fold 5 | R2: 0.581 | RMSE: 29.71 | MAPE: 14.30% | Time: 94.4s

📊 FINAL SUMMARY: FULL_SPATIAL_WITH_ABLATION
Total Time: 366.68s
-----------------------------------
R²:    0.5943 ± 0.0274
MAE:   22.17
RMSE:  30.44
NRMSE: 9.12%
MAPE:  15.27%
Acc:   0.6547
-----------------------------------

✅ Saved dataframe (with OOF predictions) to: data/corn_full_spatial_with_ablation__with_oof_preds.parquet
Columns added: pred_yield_oof, pred_fold


In [2]:
import pandas as pd
for i in pd.read_parquet("data/corn_2016_2023_processed.parquet"):
    print(i)

farm_name
longitude
latitude
pixel_id
Year
crop
yield
normalized_yield_pct
normalized_yield_z
growing_seasons_N
yield_zone
yield_stability_zone
DEM
slope_deg
aspect_deg
shape_index
soil_bulk_density
soil_clay
soil_sand
soil_silt
soil_soilorganiccarbon
prcp_04
prcp_05
prcp_06
prcp_07
prcp_08
prcp_09
prcp_10
prcp_season
srad_04
srad_05
srad_06
srad_07
srad_08
srad_09
srad_10
srad_season
gdd_04
gdd_05
gdd_06
gdd_07
gdd_08
gdd_09
gdd_10
gdd_season
VH_0603
VH_0607
VH_0608
VH_0612
VH_0614
VH_0615
VH_0619
VH_0620
VH_0624
VH_0626
VH_0627
VH_0701
VH_0702
VH_0706
VH_0708
VH_0709
VH_0713
VH_0714
VH_0717
VH_0717_2
VH_0720
VH_0721
VH_0725
VH_0726
VH_0729
VH_0729_2
VH_0730
VH_0801
VH_0802
VH_0806
VH_0810
VH_0810_2
VH_0811
VH_0813
VH_0818
VH_0819
VH_0822
VH_0822_2
VH_0823
VH_0825
VH_0825_2
VH_0826
VH_0830
VH_0831
VH_0903
VH_0903_2
VH_0904
VH_0906
VH_0907
VH_0911
VH_0912
VH_0915
VH_0916
VH_0918
VH_0919
VH_0923
VH_0924
VH_0928
VH_0930
VH_1003
VH_1005
VH_1006
VH_1010
VH_1012
VH_1013
VH_1015
VH_1017
VH_1